# YOLOv8 training — tuned for small, imbalanced dataset (110 train / 33 val, 4 classes).
Key tweaks vs defaults:
- `imgsz=640` (was 320) — small objects (bird/eye) need resolution.
- `optimizer='AdamW'`, `lr0=1e-3`, `cos_lr=True` — stable on tiny data.
- Heavy aug (`mosaic`, `mixup`, `copy_paste`) + `close_mosaic=20` to stabilise end of training.
- `patience=30`, `epochs=150` — let early stopping decide.
- `cache='ram'` — dataset fits, cuts epoch time.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

train_results = model.train(
    data='my_dataset_yolo/data.yaml',
    epochs=150,
    patience=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    seed=42,
    cache='ram',
    optimizer='AdamW',
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    warmup_epochs=3.0,
    cos_lr=True,
    close_mosaic=20,
    multi_scale=True,
    amp=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    erasing=0.4,
    box=7.5,
    cls=0.7,
    dfl=1.5,
    plots=True,
    project='runs/detect',
    name='train_tuned',
    exist_ok=True,
)

## Validate best checkpoint + per-class metrics
Also run TTA (`augment=True`) for extra mAP on val.

In [ ]:
from ultralytics import YOLO

best = YOLO('runs/detect/train_tuned/weights/best.pt')

metrics = best.val(
    data='my_dataset_yolo/data.yaml',
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.6,
    device=0,
    plots=True,
    save_json=True,
    augment=True,
)

names = best.names
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP75:    {metrics.box.map75:.4f}')
print(f'mean P:   {metrics.box.mp:.4f}')
print(f'mean R:   {metrics.box.mr:.4f}')
print('\nper-class mAP50-95:')
for i, ap in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f'  {names[int(i)]:<8s} {ap:.4f}')